# Exploration

En esta libreta exploramos los logs proporcionados.

Los objetivos principales son identificar la estructura de los logs y determinar el mapeo entre las columnas del reporte y las respuestas registradas en los logs. 

In [1]:
from pathlib import Path

Definimos algunas rutas a directorios importantes: 

In [2]:
root_dir = Path("../")
data_dir = root_dir / "data"

Los archivos log actuales:

In [3]:
log_files = data_dir.glob("*.log")

for file in log_files: print(file)

../data/2026-08-30.log
../data/2026-08-29.log
../data/2026-09-01.log
../data/2026-08-31.log


Para esta exploracion consideraremos el archivo `2026-09-01.log`.

In [20]:
log_path = data_dir / "2026-09-01.log"

assert log_path.exists() and log_path.is_file()

Cargamos el archivo: 

In [21]:
file = open(log_path, 'r')

Las lineas del archivo: 

In [22]:
lines = file.readlines()

In [23]:
type(lines)

list

Las primeras 20 lineas del archivo: 

In [24]:
for i, line in enumerate(lines[:20]):
    print(f'i={i} - line= {line}')

i=0 - line= 2026-09-01T00:02:39.862105Z | INFO [operation_Id=b5ab6865ca8ac9629ff2eb303637d408] | HTTP Request: http://apitools.com:8000/v3/users_admin/resetuser?sAMAccountName_requester=admsistemas520&sAMAccountName_target=520000228 "HTTP/1.1" 200

i=1 - line= 2026-09-01T00:02:41.536428Z | INFO [operation_Id=b5ab6865ca8ac9629ff2eb303637d408] | HTTP Request: GET https://admanager.retailstore.com/RestAPI/SearchUser?domainName=retailstore.com&AuthToken=[REDACTED_TOKEN]&range=2&startIndex=1&filter=%28sAMAccountName%3Aequal%3A520000228%29 "HTTP/1.1 200 "

i=2 - line= 2026-09-01T00:02:41.53718Z | INFO [operation_Id=b5ab6865ca8ac9629ff2eb303637d408] | ADManagerRawClient.get_users_list_info_from_admanager invoked

i=3 - line= Params to execute POST to SearchUser: {'domainName': 'retailstore.com', 'AuthToken': '[REDACTED_TOKEN]', 'range': 2, 'startIndex': 1, 'filter': '(sAMAccountName:equal:520000228)'}, Raw Response: {"UsersList":[{"EXTENSIONATTRIBUTE3":"C336","DISTINGUISHED_NAME":"CN=Usuario_

En las lineas anteriores observamos dos operaciones, cada una con un id particular. Ambas operaciones son del tipo `resetuser`. 

En esta primer iteracion, consideraremos unicamente las operaciones cuyo endpoint es de la forma `users_admin/resetuser`. 

De la muestra anterior observamos algunos aspectos importantes: 

- Cada log inicia con un timestmap.
- Los logs tienen un formato `<timestamp> |<log-level> [operation_Id=<id>]| <content>`
- Los logs pueden ser multilinea, es decir, su contenido puede incluir lineas vacias y terminan cuando aparece un nuevo timestamp. 


En cuanto a las operaciones de tipo `resetuser`, podemos identificar lo siguiente.

1. Entrada - solicitud de reseteo

    ```
    HTTP Request: http://apitools.com:8000/v3/users_admin/resetuser?sAMAccountName_requester=admsistemas520&sAMAccountName_target=520000228 "HTTP/1.1" 200
    ```
    
    Marca el inicio de la operación. De aquí salen el timestamp, el requester y el target.
    La presencia de `users_admin/resetuser` es lo que identifica la operación como
    reseteo; es el filtro de entrada del MVP.

2. Consultas a ADManager - datos de los usuarios

Por cada operación hay **dos** consultas `SearchUser`, una por cuenta. Cada una son
dos registros: la línea `HTTP Request: GET .../SearchUser?...` y, a continuación, un
registro `ADManagerRawClient.get_users_list_info_from_admanager invoked` que contiene
el JSON crudo de la respuesta.

**El orden de las dos consultas no es estable.** En unas operaciones se consulta
primero el target y en otras primero el requester. La cuenta a la que corresponde
cada respuesta se identifica por el parámetro `filter`, que viene URL-encoded:

```
filter=%28sAMAccountName%3Aequal%3A520000228%29
```

decodifica a `(sAMAccountName:equal:520000228)`. Alternativamente puede leerse
`SAM_ACCOUNT_NAME` dentro del propio JSON de respuesta, que es más directo.

La respuesta es **JSON válido** (comillas dobles) embebido en el texto del registro,
después de `Raw Response: `. Campos que nos interesan:

| Campo | Uso |
|---|---|
| `SAM_ACCOUNT_NAME` | identificar a quién corresponde el registro |
| `FIRST_NAME` | nombre |
| `LAST_NAME` | apellidos |
| `OFFICE` | oficina - **conserva cero inicial** (`0520`), tratar como texto |

3. Consulta a Proactivanet

```
ProactivanetRawClient, method = GET, url = Users, proactivanet_raw_response: [{'Id': 'ANON_...', ...}]
```

Solo se consulta el **requester**. El payload no es JSON: es el `repr` de una lista de
dicts de Python (comillas simples, `None`, `False`), así que no se puede parsear con
el mismo mecanismo que ADManager.

**El MVP no usa esta etapa.** Todos sus campos vienen anonimizados (`ANON_...`), así
que no aportan ninguna columna del reporte. Se documenta para saber que existe y
poder ignorarla sin dudar.

4. Ejecución del reseteo - desenlace

Dos registros consecutivos. Primero el POST:

```
HTTP Request: POST https://admanager.retailstore.com/RestAPI/ResetPwd?...&inputFormat=%5B%7B%22userPrincipalName%22%3A%22520000228%40retailstore.com%22%7D%5D "HTTP/1.1 200 "
```

Su timestamp es el `updated_at`. Luego la respuesta:

```
ADM-Raw response | status: 200 | body: [{'sAMAccountName': '520000228', ..., 'reset': 'yes', 'statusMessage': 'Password reset successful.', 'status': '1'}]
```

También en formato `repr` de Python, no JSON. Campos relevantes: `status` (`'1'` =
éxito), `statusMessage`, `reset`.

**El resultado anterior se produjo como resultado de un prompt a un LLM**. En dicho prompt, se incluyo una secuencia de logs con el mismo id como ejemplo y se solicito identificar las componentes claves del log, considerando las columas que se espera incluir en el reporte tabular. 

Como resultado, el LLM mapeo las columnas esperadas con su respectivo origen en las respuestas registradas en los logs. A continuacion se muestra el mapeo: columa &rarr; origen, propuesto por el LLM. 

| Columna | Origen |
|---|---|
| Requisitos | - sin definir, se emite vacía |
| timestamp | timestamp de la línea de entrada (etapa 1) |
| updated_at | timestamp del POST a `ResetPwd` (etapa 4) |
| id | `operation_Id` |
| solicitante | `sAMAccountName_requester` de la etapa 1 |
| target | `sAMAccountName_target` de la etapa 1 |
| acción | derivada del endpoint; en el MVP constante para `resetuser` |
| sistema | derivada del host de la etapa 4 (`admanager...` → ADManager) |
| nombre completo solicitante | `FIRST_NAME` + `LAST_NAME` del SearchUser del requester |
| nombre completo target | `FIRST_NAME` + `LAST_NAME` del SearchUser del target |
| oficina solicitante | `OFFICE` del SearchUser del requester |
| oficina target | `OFFICE` del SearchUser del target |
| resultado final | `status` / `statusMessage` de la etapa 4 |



Los resultados anteriores, han sido formalizados en el documento [`../docs/logs-format.md`](../docs/logs-format.md). Estos resultados son la base para proceder con la implementacion del MVP del proyecto, en el que unicamente consideramos operaciones de tipo `resetuser`. 

**Nota**

El modelo LLM utilizado para las componentes claves fue Haiku 4.5 de Anthropic.